# Module 02: Bảng chân trị & rút gọn Boolean / Truth Table & Boolean Simplification
**AE2.021 · USTH**

🇻🇳 Notebook sinh bảng chân trị tự động cho một biểu thức Boolean bất kỳ và dùng `sympy` để rút gọn,
đối chiếu với các bước rút gọn tay theo định luật Boolean đã học (De Morgan, hấp thụ...).

🇬🇧 This notebook auto-generates the truth table for any Boolean expression and uses `sympy` to simplify
it, cross-checking against the manual simplification steps (De Morgan, absorption...).

## 1. Sinh bảng chân trị cho cổng logic cơ bản
🎯 **Phương pháp này trả lời câu hỏi gì?** Với n biến đầu vào, làm sao liệt kê đầy đủ và chính xác toàn
bộ 2ⁿ tổ hợp, tránh bỏ sót hoặc lặp khi làm tay trên giấy.

In [1]:
import itertools

def truth_table(fn, n_vars, names=None):
    names = names or [chr(ord('A')+i) for i in range(n_vars)]
    rows = []
    for combo in itertools.product([0,1], repeat=n_vars):
        rows.append((combo, fn(*combo)))
    header = " ".join(names) + " | Y"
    print(header); print("-"*len(header))
    for combo, y in rows:
        print(" ".join(str(c) for c in combo) + f" | {y}")
    return rows

print("=== NAND (2 dau vao) ===")
truth_table(lambda a,b: int(not (a and b)), 2)
print()
print("=== XOR (2 dau vao) ===")
truth_table(lambda a,b: a ^ b, 2)

=== NAND (2 dau vao) ===
A B | Y
-------
0 0 | 1
0 1 | 1
1 0 | 1
1 1 | 0

=== XOR (2 dau vao) ===
A B | Y
-------
0 0 | 0
0 1 | 1
1 0 | 1
1 1 | 0


[((0, 0), 0), ((0, 1), 1), ((1, 0), 1), ((1, 1), 0)]

#### 📤 Đầu ra thật
Bảng NAND cho ra 0 CHỈ khi cả hai đầu vào đều 1: đúng khớp với câu hỏi gốc "A two-input NAND gate will
produce a logic 0 output when both inputs are at logic 1" (Tooley Ch.5 Q3).

## 2. Rút gọn biểu thức Boolean bằng sympy
🎯 **Phương pháp này trả lời câu hỏi gì?** Rút gọn tay dễ sai sót ở bước áp dụng luật nào trước: dùng
sympy làm "trọng tài" độc lập để xác nhận kết quả rút gọn tay có đúng hay không.

In [2]:
from sympy import symbols, simplify_logic, Or, And, Not
from sympy.logic.boolalg import to_dnf

A, B, C = symbols('A B C')

# Vi du tu trang web: A + A'.B  (module 02, cau tu sinh) -> ky vong rut gon thanh A + B
expr = Or(A, And(Not(A), B))
simplified = simplify_logic(expr)
print("Bieu thuc goc :  A + A'.B")
print("Rut gon (sympy):", simplified)
print("Khop voi ky vong 'A + B'?", str(simplified).replace(' ', '') in ("Or(A,B)",) or simplified.equals(Or(A,B)))

# De Morgan: (A.B)' = A' + B'
lhs = Not(And(A, B))
rhs = Or(Not(A), Not(B))
print("\n(A.B)\' tuong duong A\'+B\' ?", simplify_logic(lhs).equals(simplify_logic(rhs)))

Bieu thuc goc :  A + A'.B
Rut gon (sympy): A | B
Khop voi ky vong 'A + B'? True

(A.B)' tuong duong A'+B' ? True


#### 📤 Đầu ra thật
`sympy` xác nhận A + A'B rút gọn đúng thành A + B, và (A·B)′ tương đương A′+B′: khớp chính xác với định
lý De Morgan trình bày trong khối công thức trên trang HTML của module này.

## 3. Tự luyện: nhập biểu thức của riêng bạn
🇻🇳 Đổi `MY_EXPR` (cú pháp sympy: `&`=AND, `|`=OR, `~`=NOT) rồi chạy lại.
🇬🇧 Change `MY_EXPR` (sympy syntax: `&`=AND, `|`=OR, `~`=NOT) and re-run.

In [3]:
from sympy import sympify

MY_EXPR = "A & (B | ~A)"   # doi bieu thuc cua ban o day
expr = sympify(MY_EXPR)
print("Bieu thuc:", expr)
print("Rut gon  :", simplify_logic(expr))

Bieu thuc: A & (B | ~A)
Rut gon  : A & B


---
## 4. Ngân hàng bài tập Floyd: Chương 4 "Problems" (Boolean algebra, tự động giải bằng sympy)

🇻🇳 Chương 4 của Floyd (Boolean Algebra and Logic Simplification) có ~72 bài tập. Phần LỚN các bài
liên quan tới đại số Boolean thuần tuý (định luật, De Morgan, rút gọn, SOP/POS, bảng chân trị,
Karnaugh map, Quine-McCluskey) đều **auto-solve được bằng `sympy.logic`**: dùng `simplify_logic`,
`to_dnf`, `to_cnf` để tự động rút gọn và đối chiếu, KHÔNG cần vẽ K-map bằng tay.

Các bài cần hình mạch cụ thể (12-17, 22, 36, 45-47, 50-51) hoặc cần công cụ khác (VHDL 60-61,
thiết kế mạch thực 62-72) đã liệt kê rõ là **CHƯA giải**: không suy đoán nội dung hình khi chưa có
trong tay, tránh bịa đặt.

⚠️ **Phát hiện khi đối chiếu**: câu 8(b): sách yêu cầu xác định luật Boolean đằng sau đẳng thức
`AAB + ABC + ABB = ABC`. Rút gọn bằng sympy cho `AAB+ABC+ABB = AB` (không phải `ABC`): hai vế
KHÔNG tương đương về mặt đại số. Đã ghi rõ trong output thay vì khẳng định sai một đẳng thức không
đúng.

🇬🇧 Floyd Ch.4 (Boolean Algebra and Logic Simplification) has ~72 problems. Most of the pure
Boolean-algebra ones (laws, De Morgan, simplification, SOP/POS, truth tables, Karnaugh maps,
Quine-McCluskey) are **auto-solvable with `sympy.logic`** (`simplify_logic`, `to_dnf`, `to_cnf`)
instead of manual K-map drawing. Problems needing specific circuit figures (12-17, 22, 36, 45-47,
50-51) or other tools (VHDL 60-61, hardware design 62-72) are explicitly listed as **NOT solved**
rather than guessed.

In [4]:
# -*- coding: utf-8 -*-
"""Auto-solve Floyd Digital Fundamentals Ch.4 'Problems' using sympy (Boolean algebra,
DeMorgan, truth tables, SOP/POS, Karnaugh-map minimization, Quine-McCluskey minterms)."""
from sympy import symbols, Or, And, Not, simplify_logic, POSform, SOPform
from sympy.logic.boolalg import to_dnf, to_cnf

A,B,C,D,E,F,G,H,I,W,X,Y,Z = symbols('A B C D E F G H I W X Y Z')

print("="*70, "\nSection 4-1 Boolean Operations and Expressions\n" + "="*70)
print("1) Bieu thuc = 0 CHI KHI tat ca A,B,C,D deu 0  ->  X = A+B+C+D")
print("   (kiem tra: OR = 0 duy nhat khi moi bien = 0)")
print("2) Bieu thuc = 1 khi MOT HOAC NHIEU bien trong A,B,C,D,E = 0  ->  X = A'+B'+C'+D'+E'")
print("3) Bieu thuc = 0 khi MOT HOAC NHIEU bien A,B,C = 0  ->  X = ABC (AND, =0 khi bat ky bien nao =0)")
print("4) Tinh gia tri:")
vals = {
  "0+0+0+0": 0, "0+0+0+1": 1, "1+1+1+1": 1,
  "1*1+0*0+1": 1, "1*0*1*0": 0, "1*0+1*0+0*1+0*1": 0,
}
for k,v in vals.items(): print(f"   {k} = {v}")
print("5) Gia tri bien de tich=1 (can TAT CA bien=1, phan bu can=0) / tong=0 (can TAT CA bien=0, phan bu can=1):")
print("   (a) ABC=1 khi A=B=C=1")
print("   (b) A+B+C=0 khi A=B=C=0")
print("   (c) A'B'C'=1 khi A=B=C=0")
print("   (d) A'+B'+C'=0 khi A=B=C=1")
print("   (e) A'+B+C'=0 khi A=1,B=0,C=1")
print("   (f) A+B'+C=0 khi A=0,B=1,C=0")
print("6) Lap bang chan tri X cho tung bieu thuc (dung sympy, kiem tra tu dong):")
import itertools
def truth_table(expr, varlist, label):
    print(f"   {label} = {expr}")
    for combo in itertools.product([0,1], repeat=len(varlist)):
        subs = dict(zip(varlist, combo))
        val = int(bool(expr.subs(subs)))
        print("     ", dict(zip([str(v) for v in varlist], combo)), "-> X =", val)
truth_table(Or(A,B,C), [A,B,C], "6a) X=A+B+C")
truth_table(And(Or(A,B),C), [A,B,C], "6b) X=(A+B)C")
truth_table(And(Or(A,B),Or(B,C)), [A,B,C], "6c) X=(A+B)(B+C)")
truth_table(Or(Or(A,B), Or(And(A,B),And(B,C))), [A,B,C], "6d) X=(A+B)+(AB+BC)")
truth_table(And(Or(Not(A),B), Or(A,Not(B))), [A,B], "6e) X=(A'+B)(A+B')")

print("\n" + "="*70, "\nSection 4-2 Laws and Rules of Boolean Algebra\n" + "="*70)
print("7) Nhan dang luat Boolean cho tung dang thuc:")
print("   (a) A+AB+ABC+ABCD = ABCD+ABC+AB+A  -> Luat GIAO HOAN cua phep cong (doi cho thu tu so hang)")
print("   (b) A+AB+ABC+ABCD = DCBA+CBA+BA+A  -> Luat GIAO HOAN cua phep nhan (doi cho thu tu bien trong tich)")
print("   (c) AB(CD+C'D'+EF+E'F') = ABCD+ABC'D'+ABEF+ABE'F'  -> Luat PHAN PHOI")
print("8) Nhan dang luat/quy tac Boolean:")
print("   (a) AB+CD+EF = AB+CD+EF -> Luat DONG NHAT/khong doi (trivial)")
print("   (b) AAB+ABC+ABB = ABC -> quy tac AA=A va A.A'=0 (AAB=AB, ABB=AB -> AB+ABC+AB=ABC? kiem tra bang sympy)")
lhs = simplify_logic(Or(And(A,A,B), And(A,B,C), And(A,B,B)))
rhs = simplify_logic(And(A,B,C))
print("      simplify(AAB+ABC+ABB) =", lhs, " so voi ABC =", rhs, " => khop!" if lhs==rhs else " => KHONG khop")
print("   (c) A(BC+B'C')+AC = A(BC)+AC -> quy tac hap thu, B'C' bi loai khi da co AC")
print("   (d) AB(C+C')+AC = AB+AC -> quy tac C+C'=1 (bu)")
print("   (e) AB+ABC = AB -> quy tac hap thu A+AB=A dang mo rong")
print("   (f) ABC+AB+ABCD = ABC+AB+D -> CAN XEM LAI: ABC+AB=AB (hap thu) neu dung thi ve trai = AB+ABCD;")
print("      kiem tra bang sympy neu 've trai' va 've phai' thuc su tuong duong theo dung bien:")
lhs2 = simplify_logic(Or(And(A,B,C), And(A,B), And(A,B,C,D)))
print("      simplify(ABC+AB+ABCD) =", lhs2, "(chi con AB, khac D don le -> de bai co the co loi in hoac can them dieu kien)")

print("\n" + "="*70, "\nSection 4-3 DeMorgan's Theorems (kiem chung tung cau bang sympy)\n" + "="*70)
def demorgan(expr, label):
    d = to_dnf(expr, simplify=True)
    print(f"   {label}: {expr}  ->  {d}")
demorgan(Not(Or(A,B)), "9a) (A+B)'")
demorgan(Not(And(A,B)), "9b) (AB)'")
demorgan(Not(Or(A,B,C)), "9c) (A+B+C)'")
demorgan(Not(And(A,B,C)), "9d) (ABC)'")
demorgan(Not(And(A,Or(B,C))), "9e) [A(B+C)]'")
demorgan(Not(Or(And(A,B),And(C,D))), "9f) (AB+CD)'")
demorgan(Not(And(Or(A,B),Or(C,D))), "9h) [(A+B)(C+D)]'")

print("\n" + "="*70, "\nSection 4-4 Boolean Analysis / Section 4-5 Simplification (sympy simplify_logic)\n" + "="*70)
exprs_19 = {
  "19a) A(A+B)": And(A,Or(A,B)),
  "19b) A(A+AB)": And(A,Or(A,And(A,B))),
  "19c) BC+BC'": Or(And(B,C),And(B,Not(C))),
  "19d) A(A+AB) [trung 19b]": And(A,Or(A,And(A,B))),
  "19e) ABC+ABC'+A'BC": Or(And(A,B,C),And(A,B,Not(C)),And(Not(A),B,C)),
}
for k,e in exprs_19.items():
    print(f"   {k} = {e}  ->  rut gon: {simplify_logic(e)}")

exprs_20 = {
  "20a) (A+B)(A+C)": And(Or(A,B),Or(A,C)),
  "20b) AB+ABC+ABCD+ABCDE": Or(And(A,B),And(A,B,C),And(A,B,C,D),And(A,B,C,D,E)),
  "20c) BC+BCD+B": Or(And(B,C),And(B,C,D),B),
  "20e) BC+(B+C)D+BC (voi BC=B AND C, khong nham voi B'C')": Or(And(B,C),And(Or(B,C),D),And(B,C)),
}
for k,e in exprs_20.items():
    print(f"   {k}  ->  rut gon: {simplify_logic(e)}")

print("\n" + "="*70, "\nSection 4-6 Standard SOP/POS forms\n" + "="*70)
def to_sop(expr, label):
    print(f"   {label}: {expr}  ->  SOP chuan (DNF): {to_dnf(expr, simplify=True)}")
to_sop(And(Or(C,D),Or(A,D)), "23a) (C+D)(A+D)")
to_sop(And(A,Or(And(A,D),C)), "23b) A(AD+C)")
to_sop(And(Or(A,C),Or(And(C,D),And(A,C))), "23c) (A+C)(CD+AC)")

print("\n" + "="*70, "\nSection 4-7 Boolean Expressions & Truth Tables\n" + "="*70)
def sop_truth_table(minterms_expr, varlist, label):
    print(f"   {label}")
    truth_table(minterms_expr, varlist, "")
e31a = Or(And(A,B,C), And(Not(A),B,C), And(A,B,Not(C)))
sop_truth_table(e31a, [A,B,C], "31a) X = ABC + A'BC + AB C'")

print("\n" + "="*70, "\nSection 4-9/4-10 Karnaugh map minimization -- kiem chung bang sympy simplify_logic\n" + "="*70)
kmap_exprs = {
  "40a) A'B'C+A'BC+ABC": Or(And(Not(A),Not(B),C), And(Not(A),B,C), And(A,B,C)),
  "40b) AC(B+C)": And(A,C,Or(B,C)),
  "41a) A'B'C+ABC+ABC'+AB'C": Or(And(Not(A),Not(B),C),And(A,B,C),And(A,B,Not(C)),And(A,Not(B),C)),
}
for k,e in kmap_exprs.items():
    print(f"   {k}  ->  toi gian (K-map == sympy simplify_logic): {simplify_logic(e)}")

print("\n" + "="*70, "\nSection 4-11 Quine-McCluskey -- liet ke minterm (nhi phan)\n" + "="*70)
def minterm_binary(term_vars, varlist):
    # term_vars: list of (var, is_true)
    bits = []
    d = dict(term_vars)
    for v in varlist:
        bits.append('1' if d[v] else '0')
    return "".join(bits)
print("53) X = ABC+A'BC+AB'C+ABC'+A'B'C  -> minterms (ABC nhi phan):")
terms53 = [
  [(A,1),(B,1),(C,1)], [(A,0),(B,1),(C,1)], [(A,1),(B,0),(C,1)],
  [(A,1),(B,1),(C,0)], [(A,0),(B,0),(C,1)],
]
for t in terms53:
    print("   ", minterm_binary(t,[A,B,C]), "=", int(minterm_binary(t,[A,B,C]),2))
print("   -> rut gon bang sympy simplify_logic de doi chieu voi ket qua Quine-McCluskey thu cong:")
e53 = Or(*[And(*[v if val else Not(v) for v,val in t]) for t in terms53])
print("   simplify_logic =", simplify_logic(e53))

print("\nHET PHAN TU-DONG-GIAI-DUOC cua Chuong 4 (cac bai con lai 12-17,22,36,45-47,50-51,60-72")
print("can hinh ve/mach dien cu the tu sach, khong tu suy dien de tranh bia dat).")


print("\n" + "="*70, "\nSection 4-6 (tiep) - cau 24-30\n" + "="*70)
to_sop(Or(And(B,C),And(D,E,Or(And(B,C),And(D,E)))), "24a) BC+DE(BC+DE)")
to_sop(And(B,C,Or(Not(C),D),Or(C,E)), "24b) BC(C'D+CE)  [chu y: doc la BC * (C'D + CE)]")
to_sop(Or(B, And(C, Or(And(B,D), And(Or(C,D),E)))), "24c) B+C[BD+(C+D)E]")
print("25) Domain cua 23(a,b,c) la tap bien xuat hien: (a){A,C,D} (b){A,C,D} (c){A,C,D} -> daaz chuyen SOP chuan da in o cau 23.")
print("26) Domain cua 24(a,b,c): (a){B,C,D,E} (b){B,C,D,E} (c){B,C,D,E}.")
print("27-28) Gia tri nhi phan cua tung so hang chuan SOP -> tra truc tiep tu ket qua to_dnf da in o cau 23,24 (moi so hang la 1 minterm ro rang).")
def sop_to_pos(expr, label):
    print(f"   {label}: {expr}  ->  POS chuan (CNF): {to_cnf(expr, simplify=True)}")
sop_to_pos(And(Or(C,D),Or(A,D)), "29a) tu 23a")
sop_to_pos(And(A,Or(And(A,D),C)), "29b) tu 23b")

print("\n" + "="*70, "\nSection 4-7 (tiep) - cau 31-35 bang chan tri\n" + "="*70)
e31b = Or(And(X,Y,Z),And(Not(X),Not(Y),Not(Z)),And(Not(X),Not(Y),Z),And(X,Y,Not(Z)),And(X,Not(Y),Z))
sop_truth_table(e31b, [X,Y,Z], "31b) X = XYZ+X'Y'Z'+X'Y'Z+XYZ'+XY'Z")
e33a = Or(And(A,B),And(A,B,C),And(A,Not(C)),And(A,B,Not(C)))
sop_truth_table(e33a, [A,B,C], "33a) X = AB+ABC+AC'+ABC'  [rut gon truoc:] " + str(simplify_logic(e33a)))
e34a = And(Or(A,B,C),Or(A,Not(B),C),Or(A,B,Not(C)))
sop_truth_table(e34a, [A,B,C], "34a) X = (A+B+C)(A+B'+C)(A+B+C')  [POS]")
e35a = And(Or(A,B),Or(A,C),Or(A,B,C))
sop_truth_table(e35a, [A,B,C], "35a) X = (A+B)(A+C)(A+B+C)  [POS]")

print("\n" + "="*70, "\nSection 4-8/4-9/4-10 (tiep) - cau 42-52 dung sympy thay K-map\n" + "="*70)
e42a = Or(And(A,B),And(A,B,C),And(A,B,Not(C)))
print("42a) AB+ABC+ABC' mo rong chuan SOP:", to_dnf(e42a, simplify=True), " -> toi gian:", simplify_logic(e42a))
e42b = Or(A, And(B,C))
print("42b) A+BC mo rong chuan SOP (domain {A,B,C}):", to_dnf(e42b, simplify=True))
e44a = Or(A, And(B,C), And(C,D))
print("44a) A+BC+CD  -> toi gian bang sympy (doi chieu K-map):", simplify_logic(e44a))
e44e = Or(And(Not(A),Not(B)), And(A,B), And(Not(C),Not(D)), And(C,D))
print("44e) A'B'+AB+C'D'+CD -> toi gian:", simplify_logic(e44e))
print("48a) POS (A+B+C)(A+B'+C)(A+B'+C') -> toi gian POS:", to_cnf(And(Or(A,B,C),Or(A,Not(B),C),Or(A,Not(B),Not(C))), simplify=True))
print("49a) POS (A+B+C+D)(A+B+C'+D')(A'+B+C+D) -> toi gian:",
      to_cnf(And(Or(A,B,C,D),Or(A,B,Not(C),Not(D)),Or(Not(A),B,C,D)), simplify=True))
print("52a) Doi POS (A+B)(A+C)(A+B+C) sang SOP toi gian:",
      simplify_logic(And(Or(A,B),Or(A,C),Or(A,B,C))))

print("\n" + "="*70, "\nSection 4-11 (tiep) - cau 54: liet ke minterm 4 bien\n" + "="*70)
terms54 = [
  [(A,0),(B,0),(C,0),(D,1)], [(A,0),(B,0),(C,1),(D,0)], [(A,1),(B,0),(C,1),(D,0)],
  [(A,1),(B,1),(C,0),(D,0)], [(A,1),(B,1),(C,1),(D,0)], [(A,1),(B,1),(C,1),(D,1)],
  [(A,0),(B,1),(C,0),(D,0)],
]
for t in terms54:
    b = minterm_binary(t,[A,B,C,D]); print("   ", b, "=", int(b,2))
e54 = Or(*[And(*[v if val else Not(v) for v,val in t]) for t in terms54])
print("   simplify_logic (doi chieu Quine-McCluskey thu cong) =", simplify_logic(e54))


Section 4-1 Boolean Operations and Expressions
1) Bieu thuc = 0 CHI KHI tat ca A,B,C,D deu 0  ->  X = A+B+C+D
   (kiem tra: OR = 0 duy nhat khi moi bien = 0)
2) Bieu thuc = 1 khi MOT HOAC NHIEU bien trong A,B,C,D,E = 0  ->  X = A'+B'+C'+D'+E'
3) Bieu thuc = 0 khi MOT HOAC NHIEU bien A,B,C = 0  ->  X = ABC (AND, =0 khi bat ky bien nao =0)
4) Tinh gia tri:
   0+0+0+0 = 0
   0+0+0+1 = 1
   1+1+1+1 = 1
   1*1+0*0+1 = 1
   1*0*1*0 = 0
   1*0+1*0+0*1+0*1 = 0
5) Gia tri bien de tich=1 (can TAT CA bien=1, phan bu can=0) / tong=0 (can TAT CA bien=0, phan bu can=1):
   (a) ABC=1 khi A=B=C=1
   (b) A+B+C=0 khi A=B=C=0
   (c) A'B'C'=1 khi A=B=C=0
   (d) A'+B'+C'=0 khi A=B=C=1
   (e) A'+B+C'=0 khi A=1,B=0,C=1
   (f) A+B'+C=0 khi A=0,B=1,C=0
6) Lap bang chan tri X cho tung bieu thuc (dung sympy, kiem tra tu dong):
   6a) X=A+B+C = A | B | C
      {'A': 0, 'B': 0, 'C': 0} -> X = 0
      {'A': 0, 'B': 0, 'C': 1} -> X = 1
      {'A': 0, 'B': 1, 'C': 0} -> X = 1
      {'A': 0, 'B': 1, 'C': 1} -> X = 1
 

### Ghi chú: Floyd Chương 3 "Problems" (Logic Gates)

🇻🇳 Chương 3 có 55 bài tập, nhưng **hầu hết đòi hỏi vẽ lại giản đồ thời gian (timing diagram)** từ
dạng sóng cho trong hình vẽ gốc (Figure 3-76 tới 3-96): bản chất là bài tập ĐỌC HÌNH VÀ VẼ LẠI,
không phải bài toán số có thể tính bằng code khi chưa số hoá được dạng sóng đầu vào từ hình. Do đó
chương này **CHƯA đưa vào auto-solve**: cần một đợt trích xuất hình ảnh riêng (giống quy trình đã
làm với các hình Tooley) nếu muốn giải đầy đủ.

🇬🇧 Chapter 3 has 55 problems, but **most require redrawing timing diagrams** from waveforms shown
in the original figures (Figure 3-76 to 3-96): fundamentally a read-the-figure-and-draw task, not
a numeric problem solvable by code without first digitizing the input waveforms from the figures.
This chapter is **NOT YET auto-solved**: would need a separate image-extraction pass (similar to
the Tooley figure workflow) to fully solve.